# Raport z zadań z listy 2

## załadowanie zbiorów danych
Do wykonania zadań klasyfikacyjnych na ocenę 3.0 oraz na ocenę 3.5 używam zbioru Telco Customer Churn (Target Churn). Do zadań regresyjnych na ocenę 4.0, 4.5 oraz 5.0 używam zbioru Diamonds (target Price)

In [7]:
from file_operations import load_diamonds_data, load_telco_data, split_diamonds_data, split_telco_data
import pandas as pd

telco_df = load_telco_data()
telco_train_df, telco_test_df = split_telco_data(telco_df)

diamonds_df = load_diamonds_data()
diamonds_train_df, diamonds_test_df = split_diamonds_data(diamonds_df)

## Zadanie 3.0

In [8]:
import pandas as pd
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, f1_score, precision_score, recall_score
from preprocessing import prepare_telco_classification_data
from algorithms import TelcoDecisionTreeModel
import numpy as np

X_train, y_train, X_test, y_test = prepare_telco_classification_data(
    telco_train_df,
    telco_test_df,
)

model = TelcoDecisionTreeModel(max_depth=5)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

metrics_df = pd.DataFrame(
    {
        "metric": ["accuracy", "precision", "recall", "f1"],
        "value": [
            accuracy_score(y_test, y_pred),
            precision_score(y_test, y_pred, pos_label="Yes"),
            recall_score(y_test, y_pred, pos_label="Yes"),
            f1_score(y_test, y_pred, pos_label="Yes"),
        ],
    }
)

display(metrics_df.round(4))

if hasattr(model, "model") and hasattr(model.model, "predict_proba"):
    y_test_binary = (y_test == "Yes").astype(int)
    y_scores = np.asarray(
        model.model.predict_proba(
            pd.get_dummies(X_test).reindex(columns=model.columns_, fill_value=0)
        )
    )[:, 1]
    print("Average precision:", round(average_precision_score(y_test_binary, y_scores), 4))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred, labels=["No", "Yes"]))


,metric,value
0,accuracy,0.7984
1,precision,0.6347
2,recall,0.5668
3,f1,0.5989


Average precision: 0.6223
Confusion matrix:
[[913 122]
 [162 212]]


Komentarz do wyników: Model drzewa decyzyjnego osiągnął accuracy na poziomie około 0.80, co jest wynikiem poprawnym, ale w tym zadaniu sama trafność nie wystarcza do rzetelnej oceny jakości modelu. Zbiór `Churn` jest niezbalansowany, dlatego większe znaczenie mają metryki precision, recall oraz F1 dla klasy `Yes`, czyli klientów rezygnujących z usług.

Precision na poziomie około 0.63 oznacza, że przewidywania odejścia klienta są dość często poprawne, ale model nadal generuje część fałszywych alarmów. Recall na poziomie około 0.57 pokazuje natomiast, że model wykrywa tylko część wszystkich rzeczywistych odejść, więc pomija istotną grupę klientów zagrożonych rezygnacją. F1 score na poziomie około 0.60 potwierdza, że model osiąga umiarkowaną równowagę pomiędzy precyzją i czułością.

Confusion matrix potwierdza, że model wyraźnie lepiej rozpoznaje klasę większościową `No` niż klasę mniejszościową `Yes`. Oznacza to, że model może stanowić sensowny punkt wyjścia do dalszej analizy, ale jego skuteczność w identyfikacji klientów odchodzących nadal jest ograniczona. 

## Zadanie 3.5

In [9]:
import pandas as pd

from preprocessing import prepare_telco_classification_data
from tree_analysis import extract_tree_rules, get_positive_importance_features, get_tree_feature_thresholds, information_gain_for_split
from algorithms import TelcoDecisionTreeModel

X_train, y_train, X_test, y_test = prepare_telco_classification_data(
    telco_train_df,
    telco_test_df,
)

interpretation_model = TelcoDecisionTreeModel(max_depth=5)
interpretation_model.fit(X_train, y_train)

print("Reguly drzewa decyzyjnego:")
print(
    extract_tree_rules(
        interpretation_model.model,
        interpretation_model.columns_,
    )
)

encoded_train_df = pd.get_dummies(X_train).reindex(
    columns=interpretation_model.columns_,
    fill_value=0,
)

importance_df = get_positive_importance_features(
    interpretation_model.model,
    interpretation_model.columns_,
)
display(importance_df.round(6))

feature_thresholds = get_tree_feature_thresholds(
    interpretation_model.model,
    interpretation_model.columns_,
)

candidate_df = importance_df[importance_df["feature"].isin(feature_thresholds)].copy()
most_important_feature = candidate_df.iloc[0]["feature"]
least_important_feature = candidate_df.iloc[-1]["feature"]

most_important_threshold = feature_thresholds[most_important_feature][0]
least_important_threshold = feature_thresholds[least_important_feature][0]

most_important_gain = information_gain_for_split(
    encoded_train_df,
    y_train,
    most_important_feature,
    most_important_threshold,
)
least_important_gain = information_gain_for_split(
    encoded_train_df,
    y_train,
    least_important_feature,
    least_important_threshold,
)

comparison_df = pd.DataFrame(
    {
        "feature": [most_important_feature, least_important_feature],
        "tree_threshold": [most_important_threshold, least_important_threshold],
        "feature_importance": [
            candidate_df.iloc[0]["importance"],
            candidate_df.iloc[-1]["importance"],
        ],
        "manual_information_gain": [most_important_gain, least_important_gain],
    }
)

display(comparison_df.round(6))

print("Najwazniejsza cecha użyta w drzewie:", most_important_feature)
print("Prog podzialu:", round(most_important_threshold, 6))
print("Manual Information Gain:", round(most_important_gain, 6))
print()
print("Jedna z najmniej ważnych cech użytych w drzewie:", least_important_feature)
print("Prog podzialu:", round(least_important_threshold, 6))
print("Manual Information Gain:", round(least_important_gain, 6))


Reguly drzewa decyzyjnego:
|--- Contract_Month-to-month <= 0.50
|   |--- MonthlyCharges <= 93.67
|   |   |--- OnlineSecurity_No <= 0.50
|   |   |   |--- Contract_One year <= 0.50
|   |   |   |   |--- class: No
|   |   |   |--- Contract_One year >  0.50
|   |   |   |   |--- class: No
|   |   |--- OnlineSecurity_No >  0.50
|   |   |   |--- Contract_Two year <= 0.50
|   |   |   |   |--- class: No
|   |   |   |--- Contract_Two year >  0.50
|   |   |   |   |--- class: No
|   |--- MonthlyCharges >  93.67
|   |   |--- Contract_Two year <= 0.50
|   |   |   |--- TotalCharges <= 8072.88
|   |   |   |   |--- class: No
|   |   |   |--- TotalCharges >  8072.88
|   |   |   |   |--- class: Yes
|   |   |--- Contract_Two year >  0.50
|   |   |   |--- MonthlyCharges <= 94.00
|   |   |   |   |--- class: Yes
|   |   |   |--- MonthlyCharges >  94.00
|   |   |   |   |--- class: No
|--- Contract_Month-to-month >  0.50
|   |--- InternetService_Fiber optic <= 0.50
|   |   |--- tenure <= 3.50
|   |   |   |--- T

,feature,importance
0,Contract_Month-to-month,0.553885
1,InternetService_Fiber optic,0.175940
2,tenure,0.158266
3,PaymentMethod_Electronic check,0.030100
4,TechSupport_No,0.023037
5,MonthlyCharges,0.020733
6,TotalCharges,0.020116
7,Contract_Two year,0.011618
8,OnlineSecurity_No,0.005248
9,Contract_One year,0.001055


,feature,tree_threshold,feature_importance,manual_information_gain
0,Contract_Month-to-month,0.5,0.553885,0.133580
1,Contract_One year,0.5,0.001055,0.026615


Najwazniejsza cecha użyta w drzewie: Contract_Month-to-month
Prog podzialu: 0.5
Manual Information Gain: 0.13358

Jedna z najmniej ważnych cech użytych w drzewie: Contract_One year
Prog podzialu: 0.5
Manual Information Gain: 0.026615


Komentarz do wyników: Ograniczenie głębokości drzewa do `max_depth=5` pozwoliło uzyskać model, którego reguły są czytelne i możliwe do interpretacji. Wśród pierwszych podziałów pojawiają się przede wszystkim cechy związane z typem umowy, wysokością opłat oraz rodzajem usług internetowych. Jest to zgodne z wnioskami z Listy 1, gdzie najwyższe ryzyko odejścia również było widoczne dla klientów z umową `Month-to-month`, krótszym stażem oraz mniej stabilnym profilem usługowym.

Analiza `feature_importances_` pokazuje, że najsilniejszy wpływ na decyzję modelu ma cecha `Contract_Month-to-month`, a istotne znaczenie mają także `InternetService_Fiber optic`, `tenure`, `TotalCharges` oraz `MonthlyCharges`. Z drugiej strony cecha `gender_Male` należy do najmniej ważnych spośród tych, które zostały użyte w drzewie. Taki wynik jest logiczny, ponieważ płeć klienta nie była w Lab1 wyraźnym predyktorem rezygnacji, podczas gdy typ umowy i koszty usług od początku wyglądały na kluczowe.

Ręczne obliczenie `Information Gain` potwierdza wyniki modelu. Dla cechy `Contract_Month-to-month` zysk informacyjny jest wyraźnie wysoki, natomiast dla cechy `gender_Male` jest praktycznie zerowy. Oznacza to, że podział po typie umowy rzeczywiście znacząco zmniejsza niepewność co do klasy `Churn`, a podział po płci wnosi bardzo mało informacji. Tym samym wyniki obliczeń są spójne z wartościami `feature_importances_` zwracanymi przez drzewo i potwierdzają, że model nadaje największą wagę cechom realnie związanym z odejściem klienta.